# This is a Voting Ensemble of Incrementally Trained Models on The US Accidents Data Set

**The Notebook is a part of the US Accidents Analysis Project**

**Link - https://www.kaggle.com/work/collections/18305074**

**This is the Voting Ensemble**

The answers will be reported as the imbalanced classification report of the Ensemble.

Do Note, the best model has been retrained at the end;
and the latest versions of the notebook will continue to report the same only. 
All the testing has been done privatelty.

Anyone interested in the model or any other steps if free to fork the notebook and try stuff out.
    
The **Voting Ensemble** Model's Imbalanced Classification Report -

                       pre       rec       spe        f1       geo       iba       sup
    
              1      0.000     0.000     1.000     0.000     0.000     0.000     20220
              2      0.796     1.000     0.000     0.887     0.000     0.000   1846275
              3      0.000     0.000     1.000     0.000     0.000     0.000    390162
              4      0.000     0.000     1.000     0.000     0.000     0.000     61862
    
    avg / total      0.634     0.796     0.204     0.706     0.000     0.000   2318519

The Following Models were used for the Ensemble: 

Logistic Regression - https://www.kaggle.com/code/animeguylrn/us-accidents-logistic-regression-out-of-core-sgd

Modified-Huber Lossed Model - https://www.kaggle.com/code/animeguylrn/us-accidents-huber-loss-out-of-core-sgd

Linear SVM - https://www.kaggle.com/code/animeguylrn/us-accidents-svm-out-of-core-sgd

Perceptron - https://www.kaggle.com/code/animeguylrn/us-accidents-perceptron-out-of-core-sgd

*Please Note, the Warnings have been specifically left on; so that the reader is aware of any poential changes or behaviour specifications.*

In [1]:
!git clone --filter=blob:none --no-checkout "https://github.com/Paras-GaurLRN/US-Accidents-EDA-Ensemble-Models.git"
%cd "US-Accidents-EDA-Ensemble-Models"
!git sparse-checkout init --cone
!git sparse-checkout set "notebooks/US Accidents - Pipelines/"
!git checkout main
%cd ..

Cloning into 'US-Accidents-EDA-Ensemble-Models'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 141 (delta 3), reused 1 (delta 1), pack-reused 136 (from 1)
Receiving objects: 100% (141/141), 20.26 KiB | 942.00 KiB/s, done.
Resolving deltas: 100% (78/78), done.
/kaggle/working/US-Accidents-EDA-Ensemble-Models
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 13 (delta 1), reused 0 (delta 0), pack-reused 10 (from 1)
Receiving objects: 100% (13/13), 17.92 MiB | 18.10 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (13/13), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
/kaggle/working


In [2]:
TRAIN_FILE = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/US_Accidents_train.csv'
TEST_FILE = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/US_Accidents_test.csv'
data_path = '/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines'
transformers_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/Transformers.pkl'

In [3]:
LogisticRegression_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-logistic-regression-out-of-core-sgd/LogisticRegression.pkl'
ModifiedHuberLossed_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-huber-loss-out-of-core-sgd/ModifiedHuberLossed.pkl'
LinearSVM_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-svm-out-of-core-sgd/LinearSVM.pkl'
Perceptron_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-perceptron-out-of-core-sgd/Perceptron.pkl'

In [4]:
import sys

sys.path.append(data_path)

# To ensure that we can import the pipe

In [5]:
print("### Libraires ###\n")
with open(f'{data_path}/Libraries.txt') as LibrariesTXT:
    for line in LibrariesTXT.readlines():
        print(line)

### Libraires ###

scikit-learn

imbalanced-learn

feature-engine


In [6]:
print("### Imports ###\n")
with open(f'{data_path}/Imports.txt') as ImportsTXT:
    for line in ImportsTXT.readlines():
        print(line)

### Imports ###

from warnings import warn

from sklearn.base import (BaseEstimator, TransformerMixin, clone)

from sklearn.utils._param_validation import StrOptions

from imblearn.base import BaseSampler

from sklearn.utils.validation import check_is_fitted

from sklearn.compose import ColumnTransformer

from feature_engine.datetime import DatetimeFeatures

from feature_engine.outliers import ArbitraryOutlierCapper

from imblearn.pipeline import Pipeline as IMBPipe

import pandas as pd

import numpy as np


In [7]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 5.1 MB/s eta 0:00:00


In [8]:
!pip list | grep -E "numpy|pandas|feature-engine|imbalanced-learn|scikit-learn"

geopandas                                1.1.3
imbalanced-learn                         0.14.1
numpy                                    2.0.2
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
scikit-learn                             1.6.1
sklearn-pandas                           2.2.0


**Note: Target column = Severity**

# The Pre-Requisites

In [9]:
import numpy as np
import pandas as pd
import warnings
from joblib import load, dump
from sklearn.base import (BaseEstimator, ClassifierMixin)
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from pipeline import (AnomalyCleaner, DateTimeFeatureEngineer, ColumnDropper, Illuminator, OutOfCoreNumericalImputer)
from sklearn.linear_model import SGDClassifier

**The Column Map**

*Pre-categories encoded for simplicity, OHE makes the final result a bit larger*
![Column Map](https://i.postimg.cc/zvyyYypX/Model-1.png)

In [10]:
from sklearn import set_config
set_config(transform_output='pandas')

# Loading The Transformers

In [11]:
Transformers = load(transformers_path)

In [12]:
for Transformer in Transformers.items():
    print(f'{Transformer[0]} : {Transformer[1]}',end='\n\n')

T1_AC : AnomalyCleaner()

T2_DTFE : DateTimeFeatureEngineer()

T3_IL : Illuminator()

T4_CD : ColumnDropper(columns=['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                       'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                       'Traffic_Calming', 'Traffic_Signal', 'Start_Lat',
                       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
                       'Street', 'City', 'Country', 'Timezone', 'Description',
                       'Zipcode', 'Airport_Code', 'Weather_Timestamp',
                       'Wind_Chill(F)'])

T5_SIM : ColumnTransformer(n_jobs=-1, remainder='passthrough',
                  transformers=[('num',
                                 OutOfCoreNumericalImputer(columns=['Temperature(F)',
                                                                    'Humidity(%)',
                                                                    'Pressure(in)',
                                                       

In [13]:
T1_AC = Transformers['T1_AC']
T2_DTFE = Transformers['T2_DTFE']
T3_IL = Transformers['T3_IL']
T4_CD = Transformers['T4_CD']
T5_SIM = Transformers['T5_SIM']
T6_ENC = Transformers['T6_ENC']
T7_SS = Transformers['T7_SS']

# The Base-Models Hyper-Parameters

In [14]:
LogisticRegression = load(LogisticRegression_path)
LogisticRegression.get_params()

{'alpha': 0.165,
 'average': False,
 'class_weight': None,
 'early_stopping': False,
 'epsilon': 0.1,
 'eta0': 0.0,
 'fit_intercept': True,
 'l1_ratio': 0.5,
 'learning_rate': 'optimal',
 'loss': 'log_loss',
 'max_iter': 1000,
 'n_iter_no_change': 5,
 'n_jobs': -1,
 'penalty': 'elasticnet',
 'power_t': 0.5,
 'random_state': 34,
 'shuffle': True,
 'tol': 0.001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [15]:
ModifiedHuberLossed = load(ModifiedHuberLossed_path)
ModifiedHuberLossed.get_params()

{'alpha': 9.95e-09,
 'average': False,
 'class_weight': None,
 'early_stopping': False,
 'epsilon': 0.1,
 'eta0': 0.0,
 'fit_intercept': True,
 'l1_ratio': 0.8,
 'learning_rate': 'optimal',
 'loss': 'modified_huber',
 'max_iter': 1000,
 'n_iter_no_change': 5,
 'n_jobs': -1,
 'penalty': 'elasticnet',
 'power_t': 0.5,
 'random_state': 34,
 'shuffle': True,
 'tol': 0.001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [16]:
LinearSVM = load(LinearSVM_path)
LinearSVM.get_params()

{'alpha': 0.1,
 'average': False,
 'class_weight': None,
 'early_stopping': False,
 'epsilon': 0.1,
 'eta0': 0.0,
 'fit_intercept': True,
 'l1_ratio': 1.0,
 'learning_rate': 'optimal',
 'loss': 'hinge',
 'max_iter': 1000,
 'n_iter_no_change': 5,
 'n_jobs': -1,
 'penalty': 'elasticnet',
 'power_t': 0.5,
 'random_state': 34,
 'shuffle': True,
 'tol': 0.001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [17]:
Perceptron = load(Perceptron_path)
Perceptron.get_params()

{'alpha': 1,
 'average': False,
 'class_weight': None,
 'early_stopping': False,
 'epsilon': 0.1,
 'eta0': 0.0,
 'fit_intercept': True,
 'l1_ratio': 0.5,
 'learning_rate': 'optimal',
 'loss': 'perceptron',
 'max_iter': 1000,
 'n_iter_no_change': 5,
 'n_jobs': -1,
 'penalty': 'elasticnet',
 'power_t': 0.5,
 'random_state': 34,
 'shuffle': True,
 'tol': 0.001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

# The Voting Ensemble Model

The reason we needed to write a custom model is due to how scikit-learn's VotingClassifier works,
as it doesn't accept pre-trained models.

In [18]:
class VotingEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self,*,
                transformers,
                estimators,
                copy=False):
        
        self.transformers = transformers
        self.estimators = estimators
        self.copy = copy
    
    def fit(self, X=None, y=None):
        warnings.warn(
            "CustomVotingClassifier assumes that all transformations and "
            "estimators are already fitted. fit() does not train any "
            "component; it only marks this classifier as fitted.",
            UserWarning
        )

        self._is_fitted = True

        return self

    def predict(self,X):
        check_is_fitted(self, attributes=["_is_fitted"])
        
        if self.copy: X = X.copy()

        for transformer in self.transformers:
            X = transformer.transform(X)

        self.predictions = pd.DataFrame(
            {
                f'{model.__class__.__name__}' : model.predict(X) for model in self.estimators
            },
            copy=True
        )

        self.counts = pd.DataFrame(
            np.zeros(self.predictions.to_numpy().shape[0],dtype=np.uint64),
            copy=True
        )

        def majority_vote(row):
            counts = row.value_counts()
            max_count = counts.max()
    
            winners = counts[counts == max_count].index
    
            return max(winners)
    
        return self.predictions.apply(majority_vote, axis=1).to_numpy()

In [19]:
VCModel = VotingEnsembleClassifier(
    transformers = [T1_AC,T2_DTFE,T3_IL,T4_CD,T5_SIM,T6_ENC,T7_SS],
    estimators = [LogisticRegression,ModifiedHuberLossed,LinearSVM,Perceptron]
)

# Model Performance

In [20]:
from imblearn.metrics import classification_report_imbalanced

test_set = pd.read_csv(TEST_FILE,index_col='ID')

X_pred, y_true = test_set.drop(columns=['Severity']), test_set['Severity']
del test_set

VCModel.fit()

y_pred = VCModel.predict(X_pred)

print("Classification Report : ")
print(classification_report_imbalanced(y_true, y_pred, digits=3))

/tmp/ipykernel_16/3424132335.py:12: UserWarning: CustomVotingClassifier assumes that all transformations and estimators are already fitted. fit() does not train any component; it only marks this classifier as fitted.
  warnings.warn(
/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines/pipeline.py:196: UserWarning: AnomalyCleaner.transform() is meant only for testing/prediction purposes. It will strictly only cap the data. Useful if the User wishes to sample the data when training but transform at the time of testing/prediction. Which is the intended behaviour. It is interchangeable with fit_resample() if 'drop_when_training' is False. set 'issue_warning' explicitly to False in transform() if you wish to turn this warning off.
  warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Classification Report : 


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   pre       rec       spe        f1       geo       iba       sup

          1      0.000     0.000     1.000     0.000     0.000     0.000     20220
          2      0.796     1.000     0.000     0.887     0.000     0.000   1846275
          3      0.000     0.000     1.000     0.000     0.000     0.000    390162
          4      0.000     0.000     1.000     0.000     0.000     0.000     61862

avg / total      0.634     0.796     0.204     0.706     0.000     0.000   2318519



# Sinking The Model

In [21]:
dump(VCModel,'VotingEnsembleAll.pkl')

['VotingEnsembleAll.pkl']

# For Inspection

Uncomment if you wish to examine the predictions

In [22]:
# print(VCModel.predictions)

In [23]:
# print(VCModel.counts)